In [1]:
!pip install datasets==3.6.0

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 491.5/491.5 kB 7.4 MB/s eta 0:00:00
  Attempting uninstall: datasets
    Found existing installation: datasets 4.0.0
    Uninstalling datasets-4.0.0:
      Successfully uninstalled datasets-4.0.0


In [2]:
from datasets import Dataset, DatasetDict, load_dataset
from transformers import AutoTokenizer
import torch
from torch.utils.data import DataLoader
import numpy as np
import torch.nn as nn
import math
from tqdm import tqdm


# Tokenization
from tokenizers import Tokenizer
from tokenizers.models import BPE
from tokenizers.pre_tokenizers import ByteLevel
from tokenizers.trainers import BpeTrainer
from tokenizers.pre_tokenizers import Whitespace
from transformers import PreTrainedTokenizerFast
from google.colab import drive

from huggingface_hub import notebook_login

# Or use login() if you have your token string
# from huggingface_hub import login
# login(token="YOUR_HF_TOKEN")

notebook_login()

In [3]:
drive.mount('/content/drive')

Mounted at /content/drive


In [5]:
raw_dataset = load_dataset("Fsoft-AIC/the-vault-function",split_set=["train/small"],
    languages=['python'])
raw_dataset

data/train/small/python-00000-of-00002.p(…):   0%|          | 0.00/278M [00:00<?, ?B/s]

data/train/small/python-00001-of-00002.p(…):   0%|          | 0.00/276M [00:00<?, ?B/s]

Generating train_small split: 0 examples [00:00, ? examples/s]

DatasetDict({
    train_small: Dataset({
        features: ['hexsha', 'repo', 'path', 'license', 'language', 'identifier', 'return_type', 'original_string', 'original_docstring', 'docstring', 'docstring_tokens', 'code', 'code_tokens', 'short_docstring', 'short_docstring_tokens', 'comment', 'parameters', 'docstring_params'],
        num_rows: 370657
    })
})

In [10]:
raw_dataset

DatasetDict({
    train_small: Dataset({
        features: ['hexsha', 'repo', 'path', 'license', 'language', 'identifier', 'return_type', 'original_string', 'original_docstring', 'docstring', 'docstring_tokens', 'code', 'code_tokens', 'short_docstring', 'short_docstring_tokens', 'comment', 'parameters', 'docstring_params'],
        num_rows: 370657
    })
})

In [12]:
train_test = raw_dataset["train_small"].train_test_split(test_size=0.05, seed=42)

validation_train = train_test["train"].train_test_split(test_size=0.05, seed=42)

dataset = DatasetDict({
    "train": validation_train["train"],
    "validation": validation_train["test"],
    "test": train_test["test"]
})
dataset

DatasetDict({
    train: Dataset({
        features: ['hexsha', 'repo', 'path', 'license', 'language', 'identifier', 'return_type', 'original_string', 'original_docstring', 'docstring', 'docstring_tokens', 'code', 'code_tokens', 'short_docstring', 'short_docstring_tokens', 'comment', 'parameters', 'docstring_params'],
        num_rows: 334517
    })
    validation: Dataset({
        features: ['hexsha', 'repo', 'path', 'license', 'language', 'identifier', 'return_type', 'original_string', 'original_docstring', 'docstring', 'docstring_tokens', 'code', 'code_tokens', 'short_docstring', 'short_docstring_tokens', 'comment', 'parameters', 'docstring_params'],
        num_rows: 17607
    })
    test: Dataset({
        features: ['hexsha', 'repo', 'path', 'license', 'language', 'identifier', 'return_type', 'original_string', 'original_docstring', 'docstring', 'docstring_tokens', 'code', 'code_tokens', 'short_docstring', 'short_docstring_tokens', 'comment', 'parameters', 'docstring_params'],
 

In [13]:
from tokenizers import Tokenizer, decoders
from tokenizers.pre_tokenizers import ByteLevel

# 1. BPE tokenizer

tokenizer = Tokenizer(BPE(unk_token="[UNK]"))
tokenizer.pre_tokenizer = ByteLevel(add_prefix_space=False)
tokenizer.decoder = decoders.ByteLevel()  # ⭐ ОБЯЗАТЕЛЬНО

trainer = BpeTrainer(
    vocab_size=32000,
    special_tokens=["[UNK]", "[CLS]", "[SEP]", "[PAD]", "[MASK]", "[BOS]", "[EOS]"],
    initial_alphabet=ByteLevel.alphabet(),  # ⭐ обязательно для ByteLevel
)

SAMPLE_SIZE = 100_000
sample = dataset["train"].shuffle(seed=42).select(range(SAMPLE_SIZE))


def batch_iterator():
    batch_size = 1000
    for i in range(0, len(sample), batch_size):
      batch = sample[i : i + batch_size]
      yield batch["short_docstring"] + batch["original_string"]

tokenizer.train_from_iterator(batch_iterator(), trainer)


fast_tokenizer = PreTrainedTokenizerFast(
    tokenizer_object=tokenizer,
    bos_token="[BOS]",
    eos_token="[EOS]",
    unk_token="[UNK]",
    pad_token="[PAD]",
    cls_token="[CLS]",
    sep_token="[SEP]",
    mask_token="[MASK]"
)

tokenizer = fast_tokenizer



In [17]:
def tokenize_function(examples):
    max_total_length = 1024
    bos, sep, eos = tokenizer.bos_token_id, tokenizer.sep_token_id, tokenizer.eos_token_id

    # 1. Подготовка строк (быстро, без токенизатора)
    prompts = [f"# Task: {c}\n" for c in examples["short_docstring"]]
    responses = examples["original_string"]

    # 2. БАТЧЕВАЯ токенизация — один вызов вместо тысяч
    prompt_encodings = tokenizer(prompts, add_special_tokens=False)["input_ids"]
    response_encodings = tokenizer(responses, add_special_tokens=False)["input_ids"]

    concat_inputs = []
    labels = []

    for prompt_ids_raw, response_ids_raw in zip(prompt_encodings, response_encodings):
        prompt_ids = [bos] + prompt_ids_raw + [sep]
        response_ids = response_ids_raw + [eos]

        # Code > total
        if len(response_ids) > max_total_length:
            response_ids = response_ids[:max_total_length - 1]
            prompt_ids = [bos]

        # Comm + Code > total → обрезаем хвост комментария
        elif len(prompt_ids) + len(response_ids) > max_total_length:
            available_space = max_total_length - len(response_ids)
            prompt_ids = prompt_ids[-available_space:]

        input_ids = prompt_ids + response_ids
        label_ids = [-100] * len(prompt_ids) + list(response_ids)

        # Жёсткий cap на 1024 (на случай переполнения после конкатенации)
        concat_inputs.append(input_ids[:max_total_length])
        labels.append(label_ids[:max_total_length])

    return {"input_ids": concat_inputs, "labels": labels}


tokenized_datasets = dataset.map(
    tokenize_function,
    batched=True,
    batch_size=1000,        # ⭐ меньше batch_size — лучше использование памяти + параллелизм
    num_proc=4,             # ⭐ ОБЯЗАТЕЛЬНО раскомментировать
    remove_columns=dataset["train"].column_names,
    desc="Tokenizing",      # прогресс-бар
)

Tokenizing (num_proc=4):   0%|          | 0/334517 [00:00<?, ? examples/s]

Tokenizing (num_proc=4):   0%|          | 0/17607 [00:00<?, ? examples/s]

Tokenizing (num_proc=4):   0%|          | 0/18533 [00:00<?, ? examples/s]

In [18]:
from torch.nn.utils.rnn import pad_sequence

class CodeLLMDataCollator:
    def __init__(self, tokenizer):
        self.tokenizer = tokenizer
        self.pad_token_id = tokenizer.pad_token_id or tokenizer.eos_token_id

    def __call__(self, features):
        input_ids = pad_sequence(
            [torch.as_tensor(f["input_ids"], dtype=torch.long) for f in features],
            batch_first=True, padding_value=self.pad_token_id
        )
        labels = pad_sequence(
            [torch.as_tensor(f["labels"], dtype=torch.long) for f in features],
            batch_first=True, padding_value=-100
        )
        attention_mask = (input_ids != self.pad_token_id).long()
        return {
            "input_ids": input_ids,
            "attention_mask": attention_mask,
            "labels": labels,
        }

In [ ]:
EOS_ID = tokenizer.eos_token_id
MAX_LEN = 1024

def pack_examples(examples):
    """
    Конкатенирует input_ids всех примеров в один поток через EOS,
    режет на куски по MAX_LEN. Также сохраняет labels с -100 на промптах.
    """
    concat_input = []
    concat_labels = []

    for ids, lbls in zip(examples["input_ids"], examples["labels"]):
        concat_input.extend(ids)
        concat_input.append(EOS_ID)
        concat_labels.extend(lbls)
        concat_labels.append(EOS_ID)  # EOS тоже учим предсказывать

    # Округляем длину вниз до кратного MAX_LEN (хвост отбрасываем)
    total = (len(concat_input) // MAX_LEN) * MAX_LEN

    input_packs = [concat_input[i:i + MAX_LEN] for i in range(0, total, MAX_LEN)]
    label_packs = [concat_labels[i:i + MAX_LEN] for i in range(0, total, MAX_LEN)]

    return {"input_ids": input_packs, "labels": label_packs}


packed_datasets = tokenized_datasets.map(
    pack_examples,
    batched=True,
    batch_size=2000,
    num_proc=4,
    remove_columns=tokenized_datasets["train"].column_names,
    desc="Packing",
)

print(f"До packing: {len(tokenized_datasets['train'])} примеров")
print(f"После packing: {len(packed_datasets['train'])} pack'ов")
print(f"Эффективное число токенов: {len(packed_datasets['train']) * MAX_LEN:,}")

Packing (num_proc=4):   0%|          | 0/334517 [00:00<?, ? examples/s]

In [ ]:

data_collator = CodeLLMDataCollator(tokenizer=tokenizer)

# DataLoaders
train_dataloader = DataLoader(
    packed_datasets["train"],
    batch_size=32,
    shuffle=True,
    collate_fn=data_collator,
    num_workers=4,                # вместо 2
    pin_memory=True,
    persistent_workers=True,      # ⭐ workers НЕ респавнятся между эпохами
    prefetch_factor=4,
    drop_last=True,
)

test_dataloader = DataLoader(
    packed_datasets["test"],
    batch_size=16,
    shuffle=False,
    collate_fn=data_collator,
    drop_last=True,
)


val_dataloader = DataLoader(
    packed_datasets["validation"],
    batch_size=16,
    shuffle=False,
    collate_fn=data_collator,
    drop_last=True,
)

In [ ]:
class RotaryEmbedding(nn.Module):
    def __init__(self, dim, max_position_embeddings=2048, base=10000):
        super().__init__()
        # dim — это размер ОДНОЙ головы (d_head)
        self.dim = dim

        # Вычисляем угловые частоты (theta) по формуле из статьи
        inv_freq = 1.0 / (base ** (torch.arange(0, dim, 2).float() / dim))
        self.register_buffer("inv_freq", inv_freq, persistent=False)

        # Заранее генерируем сетку позиций
        t = torch.arange(max_position_embeddings, dtype=torch.float32)

        # Внешнее произведение: получаем матрицу углов для каждой позиции [max_pos, dim // 2]
        freqss = torch.outer(t, self.inv_freq)

        # Дублируем частоты, чтобы они подходили под полный размер головы dim
        emb = torch.cat((freqss, freqss), dim=-1) # [max_pos, dim]

        cos_cached = emb.cos().unsqueeze(0).unsqueeze(1)
        sin_cached = emb.sin().unsqueeze(0).unsqueeze(1)

        # Считаем и сохраняем косинусы и синусы
        self.register_buffer("cos_cached", cos_cached, persistent=False)
        self.register_buffer("sin_cached", sin_cached, persistent=False)

    def forward(self, x, seq_len):
        # Возвращаем косинусы и синусы нужной длины, адаптированные под размеры тензора Q/K
        # Форма на выходе: [1, 1, seq_len, dim]
        return (
            self.cos_cached[:, :, :seq_len, :].to(dtype=x.dtype),
            self.sin_cached[:, :, :seq_len, :].to(dtype=x.dtype)
        )

def rotate_half(x):
    x1 = x[..., :x.shape[-1] // 2]
    x2 = x[..., x.shape[-1] // 2:]
    return torch.cat((-x2, x1), dim=-1)

def apply_rotary_pos_emb(tensor, cos, sin):
    # x * cos + rotate_half(x) * sin
    return (tensor * cos) + (rotate_half(tensor) * sin)

In [ ]:
class RMSNorm(nn.Module):
    def __init__(self, d_model, eps=1e-6):
        super().__init__()
        self.eps = eps
        # Обучаемый параметр масштаба (гамма) — точно так же, как в LayerNorm
        self.weight = nn.Parameter(torch.ones(d_model))

    def forward(self, x):
        # x: [batch, seq_len, d_model]

        # 1. Считаем среднее квадратов по последней оси (dim=-1)
        # keepdim=True важен для последующего деления (бродкастинга)
        variance = x.pow(2).mean(-1, keepdim=True)

        # 2. Делим x на корень из (variance + eps) и умножаем на вес
        # Магия PyTorch: rsqrt(t) это 1 / sqrt(t) — считается на GPU за один такт!
        return x * torch.rsqrt(variance + self.eps) * self.weight

In [ ]:
import torch.nn.functional as F

class MultiHeadAttention(nn.Module):
  def __init__(self, d_model, n_heads_q=12, n_heads_kv=4):
    super().__init__()
    self.d_head = d_model // n_heads_q
    self.d_model = d_model

    self.n_heads_q = n_heads_q
    self.q_dim = n_heads_q * self.d_head
    self.kv_dim = n_heads_kv * self.d_head

    # Общий размер для одного слоя проекции
    total_dim = self.q_dim + 2 * self.kv_dim

    #
    self.qkv_proj = nn.Linear(d_model, total_dim, bias=False)

    self.fc = nn.Linear(self.q_dim, d_model, bias=False)
    self.rotary_emb = RotaryEmbedding(dim=self.d_head)

  def forward(self, x, padding_mask=None):
    # x [batch, padded_size, d_model]
    batch_size = x.size(0)
    seq_len = x.size(1)
    qkv = self.qkv_proj(x)
    q, k, v = torch.split(qkv, [self.q_dim, self.kv_dim, self.kv_dim], dim=-1)

    q = q.view(batch_size, seq_len, self.n_heads_q, self.d_head).transpose(1, 2)
    k = k.view(batch_size, seq_len, self.n_heads_kv, self.d_head).transpose(1, 2)
    v = v.view(batch_size, seq_len, self.n_heads_kv, self.d_head).transpose(1, 2)

    # ====== RoPE ADDED ======
    # Получаем матрицы поворота для текущей длины последовательности
    # Переносим их на устройство (GPU), где лежит входной тензор х
    cos, sin = self.rotary_emb(x, seq_len)

    q = apply_rotary_pos_emb(q, cos, sin)
    k = apply_rotary_pos_emb(k, cos, sin)
    # =======================================

    # attn_weights [batch, padded_size, padded_size]
    # torch.triu создает верхнетреугольную матрицу. Из-за diagonal=1 диагональ остается доступной.
    # Triangle matrix
    #causal_mask = torch.triu(torch.ones(seq_len, seq_len, dtype=torch.bool, device=x.device), diagonal=1)
    #causal_mask = causal_mask.unsqueeze(0).unsqueeze(1) # -> [1, 1, seq_len, seq_len]

    # 3. Adding padding mask
    # if padding_mask is not None:
    #     while padding_mask.dim() < 4:
    #         # Убираем одну из единичных осей, чтобы стало [16, 1, 1, 293]
    #         padding_mask = padding_mask.unsqueeze(1)

    #     mask = torch.logical_or(causal_mask, padding_mask == 0).bool()
    # else:
    #    mask = causal_mask

    # attn_weights = F.softmax(attn_weights, dim=-1)
    # attn_weights = self.dropout(attn_weights)
    # output = torch.bmm(attn_weights, v)
    # # output [batch, padded_size, d_head]
    # return output

    # Replaced hand calculation to more efficient scaled_dot_product_attention
    output = F.scaled_dot_product_attention(
        q, k, v,
        #attn_mask=~mask,
        dropout_p=0.0,
        is_causal=True,
        enable_gqa=True,
    )
    # output -> [batch, +n_heads, seq_len, d_head]

    # output -> [batch, seq_len, d_model]
    output = output.transpose(1, 2).contiguous().view(batch_size, seq_len, self.q_dim)
    # output -> [batch, seq_len, d_model]
    # Mixing heads
    output = self.fc(output)

    # output -> [batch, seq_len, d_model]
    return output # .squeeze(1)

In [ ]:
class TransformerBlock(nn.Module):
  def __init__(self, d_model = 768, dim_feedforward=2048):
    super().__init__()

    self.head = MultiHeadAttention(d_model)

    # self.linear0 = nn.Linear(d_model, d_model)

    self.linear1 = nn.Linear(d_model, dim_feedforward, bias=False)
    self.linear2 = nn.Linear(dim_feedforward, d_model, bias=False)
    self.norm1 = RMSNorm(d_model)
    self.norm2 = RMSNorm(d_model)

  def forward(self, x, padding_mask):
    # x [batch, padded_size, d_model]
    normed_x = self.norm1(x) # Pre-LN
    head_output = self.head(normed_x, padding_mask)
    # head_output -> [batch, padded_size, d_model]

    x = x + head_output
    normed_x2 = self.norm2(x)
    # [batch, padded_size, d_model] -> [batch, padded_size, dim_feedforward] -> [batch, padded_size, d_model]
    ffn_output = self.linear2(F.silu(self.linear1(normed_x2)))

    # Residual Connection
    x = x + ffn_output
    # [batch, padded_size, d_model]
    return x



In [ ]:
class JuniorTransformer(nn.Module):
  def __init__(self, vocab_size, d_model=768, num_layers=12):
    super().__init__()

    self.embedding = nn.Embedding(vocab_size, d_model)
    # self.pos_encoder = PositionalEncoding(d_model)

    self.transformer_layers = nn.ModuleList([
            TransformerBlock(d_model=d_model) for _ in range(num_layers)
    ])
    self.final_norm = RMSNorm(d_model)

    self.fc = nn.Linear(d_model, vocab_size)
    self.fc.weight = self.embedding.weight


  def forward(self, x, padding_mask):
    # x [batch, padded_size]

    # x -> [batch, padded_size, d_model]
    x = self.embedding(x)

    # x -> [batch, padded_size, d_model]
    #x = self.pos_encoder(x)

    for layer in self.transformer_layers:
      x = layer(x, padding_mask)

    # x = x.mean(dim=1) # x[:, 0, :]
    #x = x[:, -1, :]
    # x -> [batch, d_model]
    x = self.final_norm(x)
    x = self.fc(x)
    # x -> [batch, vocab_size]
    return x



In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model = JuniorTransformer(len(tokenizer))
model.to(device)

def gpt_style_init_weights(module, num_layers=12):
    if isinstance(module, nn.Linear):
        # 1. Default for all FC (Q, K, V & first layer FFN)
        std = 0.02

        # 2. УСЛОЖНЕНИЕ: Если это финальный слой в остаточной связи, уменьшаем веса
        # Мы проверяем это по имени переменной, либо можно проверять размерности.
        # В вашем TransformerBlock это слои linear0 (выход attn) и linear2 (выход ffn)
        # Если вы не уверены в именах, можно проверять их логически или передавать флаг.

        torch.nn.init.normal_(module.weight, mean=0.0, std=std)

        if module.bias is not None:
            torch.nn.init.zeros_(module.bias)

    elif isinstance(module, nn.Embedding):
        torch.nn.init.normal_(module.weight, mean=0.0, std=0.02)

    elif isinstance(module, nn.LayerNorm):
        # LN: default weight 1, bias 0
        torch.nn.init.ones_(module.weight)
        torch.nn.init.zeros_(module.bias)


model.apply(lambda m: gpt_style_init_weights(m, num_layers=12))

# Layer by layer fine-tuning weight initialization of input layers
for name, sub_module in model.named_modules():
    if name.endswith("linear2"):
        scaled_std = 0.02 / ((2 * 12) ** 0.5)
        sub_module.weight.data.normal_(mean=0.0, std=scaled_std)
    elif name.endswith(".head.fc"):
        scaled_std = 0.02 / ((2 * 12) ** 0.5)
        sub_module.weight.data.normal_(mean=0.0, std=scaled_std)

optimizer = torch.optim.AdamW(model.parameters(), lr=3e-4, betas=(0.9, 0.95), weight_decay=0.1)


num_epochs = 3
steps_per_epoch = len(train_dataloader)
total_steps = num_epochs * steps_per_epoch
warmup_steps = 1000
min_lr_ratio = 0.1

def lr_lambda(step):
    if step < warmup_steps:
        return step / warmup_steps
    progress = (step - warmup_steps) / max(1, total_steps - warmup_steps)
    return min_lr_ratio + (1 - min_lr_ratio) * 0.5 * (1 + math.cos(math.pi * progress))

scheduler = torch.optim.lr_scheduler.LambdaLR(optimizer, lr_lambda)

criterion = nn.CrossEntropyLoss()# ignore_index=tokenizer.pad_token_id)

model = torch.compile(model)

Special init applied for: transformer_layers.0.linear2 (std=0.0041)
Special init applied for: transformer_layers.1.linear2 (std=0.0041)
Special init applied for: transformer_layers.2.linear2 (std=0.0041)
Special init applied for: transformer_layers.3.linear2 (std=0.0041)
Special init applied for: transformer_layers.4.linear2 (std=0.0041)
Special init applied for: transformer_layers.5.linear2 (std=0.0041)
Special init applied for: transformer_layers.6.linear2 (std=0.0041)
Special init applied for: transformer_layers.7.linear2 (std=0.0041)
Special init applied for: transformer_layers.8.linear2 (std=0.0041)
Special init applied for: transformer_layers.9.linear2 (std=0.0041)
Special init applied for: transformer_layers.10.linear2 (std=0.0041)
Special init applied for: transformer_layers.11.linear2 (std=0.0041)


In [ ]:
def eval(model, device, val_dataloader, criterion):
  model.eval()
  total_loss = 0.0

  pbar = tqdm(val_dataloader, desc="Evaluating", leave=False)

  with torch.no_grad():
    with torch.amp.autocast('cuda', dtype=torch.bfloat16):  # ⭐

      for batch in pbar:
        input_ids = batch['input_ids'].to(device)
        labels = batch['labels'].to(device)

        padding_mask = (input_ids[:, :-1] != tokenizer.pad_token_id).unsqueeze(1).unsqueeze(2)

        x_input = input_ids[:, :-1]
        target = labels[:, 1:]
        outputs = model(x_input, padding_mask)
        loss = criterion(outputs.reshape(-1, outputs.size(-1)), target.reshape(-1))

        total_loss += loss.item()


        pbar.set_postfix(loss=f"{loss.item():.4f}")

  return total_loss / len(val_dataloader)

def train(epoch, model, device, train_dataloader, optimizer, scheduler, criterion):
  model.train()
  optimizer.zero_grad()

  total_loss = 0.0
  save_every_steps = 5000


  pbar = tqdm(train_dataloader, desc="Training", leave=False)

  for step, batch in enumerate(pbar):
    input_ids = batch['input_ids'].to(device)
    labels = batch['labels'].to(device)

    padding_mask = (input_ids[:, :-1] != tokenizer.pad_token_id).unsqueeze(1).unsqueeze(2)


    x_input = input_ids[:, :-1]
    target = labels[:, 1:]

    with torch.amp.autocast('cuda', dtype=torch.bfloat16):
      outputs = model(x_input, padding_mask)


      loss = criterion(outputs.reshape(-1, outputs.size(-1)), target.reshape(-1))

    loss.backward()

    # Рекомендуется: обрезка градиентов от взрыва (NaN)
    torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)

    if step % 500 == 0:
      print(f"loss: {(total_loss/(step + 1)):.4f} | PPL: {torch.exp(total_loss/(step + 1)):.4f}")

    optimizer.step()
    scheduler.step()
    optimizer.zero_grad()

    total_loss += loss.item()

    pbar.set_postfix(loss=f"{loss.item():.4f}")

    # epoch step save
    if step > 0 and step % save_every_steps == 0:
        checkpoint = {
            'epoch': epoch,
            'step': step,
            'model_state_dict': model.state_dict(),
            'optimizer_state_dict': optimizer.state_dict(),
            'scheduler_state_dict': scheduler.state_dict(),

            'loss': loss.item(),
        }

        torch.save(checkpoint, f"checkpoint_epoch_{epoch}_step_{step}.pt")
        print(f"\n[INF] Checkpoint save {step}")

  return total_loss / len(train_dataloader)

<>:58: SyntaxWarning: invalid escape sequence '\|'
<>:58: SyntaxWarning: invalid escape sequence '\|'
/tmp/ipykernel_1726/1065985447.py:58: SyntaxWarning: invalid escape sequence '\|'
  print(f"loss: {(total_loss/step):.4f} \| PPL: {torch.exp(total_loss/step):.4f}")


In [ ]:
torch.set_float32_matmul_precision('high')

In [ ]:
for i in range(num_epochs):
  train_loss = train(i, model, device, train_dataloader, optimizer, scheduler, criterion)
  val_loss = eval(model, device, val_dataloader, criterion)
  model_to_save = model._orig_mod if hasattr(model, '_orig_mod') else model
  checkpoint = {
            'epoch': i,
            'model_state_dict': model_to_save.state_dict(),
            'optimizer_state_dict': optimizer.state_dict(),
            'loss': val_loss,
        }

  torch.save(checkpoint, f"checkpoint_epoch_{i}.pt")
  print(f'Epoch {i}/10 --- Train loss: {train_loss:.4f} | PPL {torch.exp(train_loss):.4f} --- Val loss: {val_loss:.4f} | PPL {torch.exp(val_loss):.4f}')

<>:12: SyntaxWarning: invalid escape sequence '\|'
<>:12: SyntaxWarning: invalid escape sequence '\|'
<>:12: SyntaxWarning: invalid escape sequence '\|'
<>:12: SyntaxWarning: invalid escape sequence '\|'
/tmp/ipykernel_1726/3359014992.py:12: SyntaxWarning: invalid escape sequence '\|'
  print(f'Epoch {i}/10 --- Train loss: {train_loss:.4f} \| PPL {torch.exp(train_loss):.4f} --- Val loss: {val_loss:.4f} \| PPL {torch.exp(val_loss):.4f}')
/tmp/ipykernel_1726/3359014992.py:12: SyntaxWarning: invalid escape sequence '\|'
  print(f'Epoch {i}/10 --- Train loss: {train_loss:.4f} \| PPL {torch.exp(train_loss):.4f} --- Val loss: {val_loss:.4f} \| PPL {torch.exp(val_loss):.4f}')


AttributeError: 'RMSNorm' object has no attribute 'we'

In [ ]:
import os

# Define the path to save the model in Google Drive
save_path = "/content/drive/MyDrive/programmer_1st_epoch_checkpoint.pt"
model_to_save = model._orig_mod if hasattr(model, '_orig_mod') else model

# Save the model's state dictionary
torch.save(model_to_save.state_dict(), save_path)

print(f"Model saved successfully to: {save_path}")

In [ ]:
@torch.no_grad()
def generate(model, tokenizer, prompt, max_new_tokens=100, max_length=1024,
             temperature=0.8, top_k=40):
    model.eval()

    ids = ([tokenizer.bos_token_id]
           + tokenizer(prompt, add_special_tokens=False)["input_ids"]
           + [tokenizer.sep_token_id])

    # Защита: если промпт сам по себе слишком длинный — обрезаем его конец
    if len(ids) >= max_length:
        ids = ids[-(max_length - 1):]  # оставляем место хотя бы для 1 нового токена

    x = torch.tensor([ids], device=device)
    prompt_len = x.size(1)

    for _ in range(max_new_tokens):
        # 1) Ограничиваем контекст max_length (sliding window)
        x_cond = x if x.size(1) <= max_length else x[:, -max_length:]

        mask = torch.ones(x_cond.shape, dtype=torch.bool, device=device)
        logits = model(x_cond, mask)[:, -1, :] / temperature

        # 2) top-k
        v, _ = torch.topk(logits, top_k)
        logits[logits < v[:, [-1]]] = -float('inf')
        probs = torch.softmax(logits, dim=-1)
        next_token = torch.multinomial(probs, 1)

        if next_token.item() == tokenizer.eos_token_id:
            break

        x = torch.cat([x, next_token], dim=1)

        # 3) Жёсткий стоп по абсолютной длине
        if x.size(1) >= max_length:
            break

    # Декодируем только сгенерированную часть отдельно для наглядности
    generated_ids = x[0, prompt_len:].tolist()
    full_text = tokenizer.decode(x[0].tolist(), skip_special_tokens=True)
    gen_only = tokenizer.decode(generated_ids, skip_special_tokens=True)

    return {"full": full_text, "generated": gen_only}

In [ ]:
prompts = [
    "# Task: sort a list of integers in ascending order\n",
    "# Task: read a file line by line and return list of strings\n",
    "# Task: calculate the factorial of n\n",
]

for p in prompts:
    print("=" * 60)
    print(f"PROMPT: {p}")
    result = generate(model, tokenizer, p, max_new_tokens=150)
    print(f"--- GENERATED ---\n{result['generated']}")